# 01 - Burn-In Label Simulator

Creates the visible **Wafer-Test Fail** map and the hidden **Latent Defective** answer key for every wafer.

**Inputs**
- Gaussian parametric data on the KAI die grid (`Wafer Parametric Dataset/Modified Data/`). `DATASET = "full"` uses `*_gaussian.csv` (200 wafers per lot); `DATASET = "dev"` uses `*_first10.csv` (first 10 wafers per lot).
- WM811K wafer maps (`WM811K Dataset/wm811k_dev.npz`, 0 = off-wafer, 1 = pass, 2 = fail).

**Steps** (see `docs/BurnIn_Project_Brief.md`, section 7)
1. **Limits**: mean +/- 6 sigma per parameter, pooled over Lots 1-4 before any injection.
2. **Overlay**: a WM811K defect pattern is cleaned, stretched onto the KAI wafer, rotated (and shifted for Loc/Scratch), giving a cluster mask.
3. **Wafer-Test Fails**: 30-60% of in-cluster dies plus ~0.5% random dies get 1-2 parameters pushed outside the limits.
4. **Hidden risk**: every passing die gets p = base x m(d), where m = 10 inside/touching the cluster and decays to 1 with distance d.
5. **Latent Defective**: weighted coin flip with probability p per passing die.
6. **Clues**: 60-80% of Latent Defective dies get 1-3 parameters shifted 1-3 sigma, staying inside the limits.

**Outputs** (in `cache/sim_full/` or `cache/sim_dev/`, gitignored)
- `visible.npz`: what a fab would see and the only thing the model may use (lot, wafer, x, y, parameter_1-5, wafer_test_fail).
- `hidden.npz`: simulator answer key (cluster mask, distance, risk p, latent, clued, ...). Never a model input.
- `wafers.csv`, `limits.json`, `config.json`, `wm811k_split.npz`.

Everything is seeded. Each wafer uses its own random stream keyed by its wafer number, so a wafer's result does not change when more wafers are added (only the pooled limits would move slightly).

In [ ]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import ndimage
import matplotlib.pyplot as plt
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
COLAB_ROOT = "/content/drive/MyDrive/BurnIn_Project"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path(COLAB_ROOT)
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "burnin").is_dir())
DATA_DIR = ROOT / "Wafer Parametric Dataset" / "Modified Data"
WM_PATH = ROOT / "WM811K Dataset" / "wm811k_dev.npz"
DATASET = "full"
WAFERS_PER_LOT = {"full": 200, "dev": 10}[DATASET]
SUFFIX = {"full": "gaussian", "dev": "first10"}[DATASET]
OUT_DIR = ROOT / "cache" / f"sim_{DATASET}"
FIG_DIR = ROOT / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
PARAMS = [f"parameter_{i}" for i in range(1, 6)]
LOT_FILES = {
    "Lot_1": f"training_1_{SUFFIX}.csv",
    "Lot_2": f"training_2_{SUFFIX}.csv",
    "Lot_3": f"training_3_{SUFFIX}.csv",
    "Lot_4": f"training_4_{SUFFIX}.csv",
    "Lot_5": f"test_{SUFFIX}.csv",
}
CSV_DTYPES = {"lot": "category", "wafer": np.int32, "x": np.int16, "y": np.int16, **{p: np.float64 for p in PARAMS}}
TRAIN_LOTS = ["Lot_1", "Lot_2", "Lot_3", "Lot_4"]
TEST_LOT = "Lot_5"
assert DATA_DIR.exists() and WM_PATH.exists(), "Data not found under the project root (the folder containing src/burnin)."

## Configuration
Every simulator knob lives here. The config is saved next to the outputs so each run can be traced.

In [ ]:
CONFIG = {
    "seed": 8803,
    "limit_k_sigma": 6.0,
    "wm_overlay_lot_fraction": 0.30,
    "pattern_mix": {
        "none": 0.25, "Edge-Ring": 0.12, "Edge-Loc": 0.12, "Center": 0.10, "Loc": 0.12,
        "Scratch": 0.10, "Donut": 0.07, "Random": 0.07, "Near-full": 0.05,
    },
    "min_cluster_px": {"default": 32, "Scratch": 8},
    "shift_classes": ["Loc", "Scratch"],
    "max_shift": 0.25,
    "cluster_fail_frac": [0.30, 0.60],
    "random_fail_rate": 0.005,
    "fail_n_params": [1, 2],
    "fail_excess_sigma": [0.5, 4.0],
    "base_risk": [0.004, 0.007],
    "m_max": 10.0,
    "touch_dist": 1.7,
    "decay_len": 3.0,
    "clued_frac": [0.60, 0.80],
    "clue_n_params": [1, 3],
    "clue_sigma": [1.0, 3.0],
    "clue_margin_sigma": 0.25,
    "target_latent_rate": [0.01, 0.02],
}
assert abs(sum(CONFIG["pattern_mix"].values()) - 1.0) < 1e-9

**Notes on the knobs**
- `min_cluster_px`: WM811K maps contain scattered fails as well as the pattern, and many maps were upscaled to 64 x 64, so one original die covers several pixels. Fail regions (8-connected) smaller than 32 pixels (about 1% of the wafer) are dropped, so the cluster mask is the pattern itself. Scratch uses 8 pixels because its thin lines break into small pieces. This cuts the average cluster coverage of "none" maps from 10% to about 1% while keeping most of each real pattern. Scattered fails are instead modeled by `random_fail_rate`.
- `max_shift`: random shift of Loc/Scratch patterns, as a fraction of the wafer radius.
- Distances (`touch_dist`, `decay_len`) are in **y die pitches**. KAI's grid is 115 x 193 dies on a round wafer, so one x step is 193/115 = 1.68 y pitches; `touch_dist = 1.7` makes all 8 neighbors count as touching.
- `m(d) = 1 + (m_max - 1) * exp(-max(d - touch_dist, 0) / decay_len)`.
- `base_risk` is drawn per wafer. It was raised from the brief's first guess (0.002-0.005) to 0.004-0.007 so the overall Latent Defective rate lands in the 1-2% target once scattered WM811K fails are removed from the cluster masks.

## 1. Load parametric data

In [ ]:
def load_lot(lot):
    df = pd.read_csv(DATA_DIR / LOT_FILES[lot], dtype=CSV_DTYPES)
    assert (df["lot"] == lot).all(), f"unexpected lot values in {LOT_FILES[lot]}"
    n_w = df["wafer"].nunique()
    assert n_w == WAFERS_PER_LOT, f"{LOT_FILES[lot]}: {n_w} wafers, expected {WAFERS_PER_LOT}"
    return df.drop(columns="lot")
frames, lot_of = [], {}
for lot in LOT_FILES:
    df = load_lot(lot)
    lot_of.update(dict.fromkeys(df["wafer"].unique().tolist(), lot))
    frames.append(df)
raw = pd.concat(frames, ignore_index=True)
del frames, df
key = raw["wafer"].to_numpy(np.int64) * 100_000 + raw["x"].to_numpy(np.int64) * 1_000 + raw["y"].to_numpy(np.int64)
if not (np.diff(key) > 0).all():
    raw = raw.sort_values(["wafer", "x", "y"]).reset_index(drop=True)
del key
WAFERS = pd.Series(lot_of, name="lot").sort_index().rename_axis("wafer")
N_W = len(WAFERS)
N_DIE = len(raw) // N_W
assert N_DIE * N_W == len(raw)
TEMPLATE = raw.iloc[:N_DIE][["x", "y"]].to_numpy()
XY = raw[["x", "y"]].to_numpy().reshape(N_W, N_DIE, 2)
assert (XY == TEMPLATE[None]).all(), "every wafer must have the same die sites"
DX, DY = TEMPLATE[:, 0], TEMPLATE[:, 1]
NX, NY = int(DX.max()), int(DY.max())
DIE_ASPECT = NY / NX
VALUES = raw[PARAMS].to_numpy().reshape(N_W, N_DIE, len(PARAMS))
del raw, XY
print(f"{N_W} wafers, {N_DIE:,} dies/wafer, grid {NX} x {NY}, x pitch = {DIE_ASPECT:.3f} y pitches")
display(WAFERS.reset_index().groupby("lot")["wafer"].agg(["count", "min", "max"]))

## 2. Step 1 - Test limits (mean +/- 6 sigma, Lots 1-4, before injection)

In [ ]:
k = CONFIG["limit_k_sigma"]
train_mask_w = WAFERS.isin(TRAIN_LOTS).to_numpy()
train_vals = VALUES[train_mask_w].reshape(-1, len(PARAMS))
MU = train_vals.mean(0)
SD = train_vals.std(0, ddof=1)
LO, HI = MU - k * SD, MU + k * SD
limits = pd.DataFrame({"mean": MU, "std": SD, "low_limit": LO, "high_limit": HI}, index=PARAMS)
display(limits)
def outside_limits(v):
    return (v < LO) | (v > HI)
def max_abs_z(v):
    flat = v.reshape(-1, len(PARAMS))
    return np.maximum(flat.max(0) - MU, MU - flat.min(0)) / SD
nat = outside_limits(VALUES)
nat_table = pd.DataFrame({
    "max |z| Lots 1-4": max_abs_z(train_vals),
    "max |z| Lot 5": max_abs_z(VALUES[~train_mask_w]),
    "natural fails Lots 1-4": nat[train_mask_w].sum((0, 1)),
    "natural fails Lot 5": nat[~train_mask_w].sum((0, 1)),
}, index=PARAMS)
display(nat_table.round(3))
print(f"Natural fail rate (any parameter): {nat.any(-1).mean():.6%}")
del train_vals

## 3. WM811K: lot-disjoint split and cluster masks
WM811K lots are split once: ~30% of lots feed the simulator overlays and the rest are reserved for encoder pre-training, so no wafer map is used for both.

In [ ]:
wm = np.load(WM_PATH, allow_pickle=False)
WM_X, WM_Y, WM_LOT = wm["X"], wm["y"], wm["lot"]
rng_split = np.random.default_rng(CONFIG["seed"])
wm_lots = np.unique(WM_LOT)
rng_split.shuffle(wm_lots)
overlay_lots = wm_lots[: int(round(CONFIG["wm_overlay_lot_fraction"] * len(wm_lots)))]
WM_OVERLAY = np.isin(WM_LOT, overlay_lots)
split_counts = pd.crosstab(WM_Y, np.where(WM_OVERLAY, "overlay", "pretrain"))
display(split_counts)
assert (split_counts["overlay"] > 0).all(), "every class needs at least one overlay map"
np.savez_compressed(OUT_DIR / "wm811k_split.npz", overlay_idx=np.flatnonzero(WM_OVERLAY), pretrain_idx=np.flatnonzero(~WM_OVERLAY))
OVERLAY_POOL = {c: np.flatnonzero(WM_OVERLAY & (WM_Y == c)) for c in CONFIG["pattern_mix"]}

In [ ]:
def wm_cluster(m, min_px):
    fail = m == 2
    lab, n = ndimage.label(fail, structure=np.ones((3, 3), bool))
    if n == 0:
        return fail
    sizes = ndimage.sum(fail, lab, index=np.arange(1, n + 1))
    return np.isin(lab, 1 + np.flatnonzero(sizes >= min_px))
def wm_geometry(m):
    rows = np.flatnonzero((m > 0).any(1))
    cols = np.flatnonzero((m > 0).any(0))
    return (rows[0] + rows[-1]) / 2, (rows[-1] - rows[0] + 1) / 2, (cols[0] + cols[-1]) / 2, (cols[-1] - cols[0] + 1) / 2
U = (DX - (DX.min() + DX.max()) / 2) / ((DX.max() - DX.min() + 1) / 2)
V = (DY - (DY.min() + DY.max()) / 2) / ((DY.max() - DY.min() + 1) / 2)
def overlay_cluster(m, theta, shift, min_px):
    keep = wm_cluster(m, min_px).astype(float)
    rc, rr, cc, rcol = wm_geometry(m)
    uu, vv = U - shift[0], V - shift[1]
    c, s = np.cos(theta), np.sin(theta)
    ur, vr = c * uu + s * vv, -s * uu + c * vv
    val = ndimage.map_coordinates(keep, [rc + vr * rr, cc + ur * rcol], order=1, cval=0.0)
    return val >= 0.5

## 4. Steps 2-6 - Simulate every wafer

In [ ]:
def pick_params(rng, n, kmin, kmax):
    kk = rng.integers(kmin, kmax + 1, n)
    ranks = rng.random((n, len(PARAMS))).argsort(1).argsort(1)
    return ranks < kk[:, None]
def distance_to_cluster(cluster):
    if not cluster.any():
        return np.full(N_DIE, np.inf)
    g = np.zeros((NX, NY), bool)
    g[DX - 1, DY - 1] = cluster
    dist = ndimage.distance_transform_edt(~g, sampling=(DIE_ASPECT, 1.0))
    return dist[DX - 1, DY - 1]
def risk_multiplier(d, cfg):
    return 1.0 + (cfg["m_max"] - 1.0) * np.exp(-np.maximum(d - cfg["touch_dist"], 0.0) / cfg["decay_len"])
BITS = 1 << np.arange(len(PARAMS))
def simulate_wafer(wafer_id, values, cfg):
    rng = np.random.default_rng([cfg["seed"], int(wafer_id)])
    v = values.copy()
    classes = list(cfg["pattern_mix"])
    pattern = rng.choice(classes, p=list(cfg["pattern_mix"].values()))
    wm_idx = int(rng.choice(OVERLAY_POOL[pattern]))
    theta = rng.uniform(0, 2 * np.pi)
    shift = rng.uniform(-cfg["max_shift"], cfg["max_shift"], 2) if pattern in cfg["shift_classes"] else np.zeros(2)
    min_px = cfg["min_cluster_px"].get(str(pattern), cfg["min_cluster_px"]["default"])
    cluster = overlay_cluster(WM_X[wm_idx], theta, shift, min_px)
    q = rng.uniform(*cfg["cluster_fail_frac"])
    injected = (cluster & (rng.random(N_DIE) < q)) | (rng.random(N_DIE) < cfg["random_fail_rate"])
    idx = np.flatnonzero(injected)
    fsel = pick_params(rng, len(idx), *cfg["fail_n_params"])
    sign = rng.choice([-1.0, 1.0], size=fsel.shape)
    excess = rng.uniform(*cfg["fail_excess_sigma"], size=fsel.shape)
    pushed = MU + sign * (cfg["limit_k_sigma"] + excess) * SD
    v[idx] = np.where(fsel, pushed, v[idx])
    fail_params = np.zeros(N_DIE, np.int8)
    fail_params[idx] = (fsel * BITS).sum(1)
    wtf = outside_limits(v).any(1)
    assert wtf[injected].all()
    dist = distance_to_cluster(cluster)
    base = rng.uniform(*cfg["base_risk"])
    p = base * risk_multiplier(dist, cfg)
    p[wtf] = 0.0
    latent = rng.random(N_DIE) < p
    cf = rng.uniform(*cfg["clued_frac"])
    clued = latent & (rng.random(N_DIE) < cf)
    cidx = np.flatnonzero(clued)
    csel = pick_params(rng, len(cidx), *cfg["clue_n_params"])
    delta = rng.choice([-1.0, 1.0], size=csel.shape) * rng.uniform(*cfg["clue_sigma"], size=csel.shape) * SD
    lim = (cfg["limit_k_sigma"] - cfg["clue_margin_sigma"]) * SD
    cur = v[cidx]
    new = cur + delta
    bad = np.abs(new - MU) >= lim
    new = np.where(bad, cur - delta, new)
    new = np.clip(new, MU - lim, MU + lim)
    v[cidx] = np.where(csel, new, cur)
    clue_params = np.zeros(N_DIE, np.int8)
    clue_params[cidx] = (csel * BITS).sum(1)
    assert (outside_limits(v).any(1) == wtf).all(), "clues must not change pass/fail"
    meta = {
        "wafer": int(wafer_id), "pattern": str(pattern), "wm_idx": wm_idx, "theta_deg": float(np.degrees(theta)),
        "shift_u": float(shift[0]), "shift_v": float(shift[1]), "cluster_fail_frac": float(q),
        "base_risk": float(base), "clued_frac": float(cf),
        "n_cluster": int(cluster.sum()), "n_injected": int(injected.sum()), "n_wtf": int(wtf.sum()),
        "n_pass": int((~wtf).sum()), "n_latent": int(latent.sum()), "n_clued": int(clued.sum()),
    }
    hidden = {"cluster": cluster, "dist": dist.astype(np.float32), "risk_p": p.astype(np.float32), "latent": latent,
              "clued": clued, "injected_fail": injected, "fail_params": fail_params, "clue_params": clue_params}
    return v, wtf, hidden, meta
SIM_VALUES = np.empty_like(VALUES)
SIM_WTF = np.empty((N_W, N_DIE), bool)
HIDDEN = {}
META = []
for i, (wafer_id, lot) in enumerate(WAFERS.items()):
    v, wtf, hid, meta = simulate_wafer(wafer_id, VALUES[i], CONFIG)
    SIM_VALUES[i], SIM_WTF[i] = v, wtf
    for key, arr in hid.items():
        HIDDEN.setdefault(key, np.empty((N_W, N_DIE), arr.dtype))[i] = arr
    META.append({"lot": lot, **meta})
wafers = pd.DataFrame(META)
print(f"Simulated {len(wafers)} wafers")

## 5. Checks and summary

In [ ]:
H = HIDDEN
assert not (H["latent"] & SIM_WTF).any(), "Latent Defective dies must pass wafer test"
assert not (H["clued"] & ~H["latent"]).any()
assert (SIM_WTF | ~H["injected_fail"]).all()
untouched = ~H["injected_fail"] & ~H["clued"]
assert ((SIM_VALUES == VALUES).all(-1) | ~untouched).all(), "only injected/clued dies may change"
NEAR_DIST = 2.0
near = H["dist"] <= NEAR_DIST
def split_summary(mask_w):
    passing = ~SIM_WTF[mask_w]
    lat = H["latent"][mask_w]
    n_lat = lat.sum()
    hopeless = lat & ~H["clued"][mask_w] & ~near[mask_w]
    return {
        "wafers": int(mask_w.sum()),
        "Wafer-Test Fail %": 100 * SIM_WTF[mask_w].mean(),
        "  of which natural %": 100 * (SIM_WTF[mask_w] & ~H["injected_fail"][mask_w]).mean(),
        "Latent Defective (% of passing)": 100 * n_lat / passing.sum(),
        "Latent Defective count": int(n_lat),
        "clued (% of latent)": 100 * H["clued"][mask_w].sum() / max(n_lat, 1),
        "near cluster (% of latent)": 100 * (lat & near[mask_w]).sum() / max(n_lat, 1),
        "unclued and isolated (% of latent)": 100 * hopeless.sum() / max(n_lat, 1),
        "recall ceiling %": 100 * (1 - hopeless.sum() / max(n_lat, 1)),
    }
summary = pd.DataFrame({"Lots 1-4": split_summary(train_mask_w), "Lot 5": split_summary(~train_mask_w), "All": split_summary(np.ones(N_W, bool))})
display(summary.round(3))
rate = H["latent"].sum() / (~SIM_WTF).sum()
lo_t, hi_t = CONFIG["target_latent_rate"]
print(f"Overall Latent Defective rate {rate:.3%} -> {'within' if lo_t <= rate <= hi_t else 'OUTSIDE'} target {lo_t:.0%}-{hi_t:.0%}")

The **recall ceiling** counts a Latent Defective die as catchable if it is clued or within `NEAR_DIST` = 2 die pitches of a cluster (the same definition as `burnin.metrics.recall_ceiling(ws, near=2.0)` in notebooks 02 and 05); all other latent dies are uncatchable. A model scoring close to 100% recall at a small burn-in fraction would suggest leakage.

In [ ]:
by_pattern = wafers.groupby("pattern").agg(
    wafers=("wafer", "count"), cluster_dies=("n_cluster", "mean"), wtf=("n_wtf", "mean"),
    latent=("n_latent", "mean"), clued=("n_clued", "mean"))
by_pattern["wtf %"] = 100 * by_pattern["wtf"] / N_DIE
by_pattern["latent % of passing"] = 100 * wafers.groupby("pattern").apply(lambda g: g["n_latent"].sum() / g["n_pass"].sum(), include_groups=False)
display(by_pattern.round(2))
display(wafers.head(10))

## 6. Example wafers

In [ ]:
def to_grid(vals, fill=np.nan):
    g = np.full((NX, NY), fill, float)
    g[DX - 1, DY - 1] = vals
    return g
def show(ax, vals, title, **kw):
    ax.imshow(to_grid(vals), aspect=DIE_ASPECT, interpolation="nearest", **kw)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
examples = wafers.drop_duplicates("pattern").head(4).index.to_list()
fig, axes = plt.subplots(len(examples), 5, figsize=(15, 3.2 * len(examples)))
wm_cmap = plt.matplotlib.colors.ListedColormap(["white", "#9ecae1", "#d62728"])
for r, i in enumerate(examples):
    w = wafers.loc[i]
    ax = axes[r]
    ax[0].imshow(WM_X[w.wm_idx], cmap=wm_cmap, vmin=0, vmax=2, interpolation="nearest")
    ax[0].set_title(f"WM811K source: {w.pattern}", fontsize=9)
    ax[0].axis("off")
    show(ax[1], H["cluster"][i].astype(float), f"Cluster mask (hidden), wafer {w.wafer}", cmap="Greys", vmin=0, vmax=1)
    show(ax[2], SIM_WTF[i].astype(float), f"Wafer-Test Fail map (visible): {w.n_wtf} dies", cmap="Reds", vmin=0, vmax=1)
    show(ax[3], H["risk_p"][i], "Hidden risk p", cmap="magma")
    show(ax[4], SIM_WTF[i] * 0.25, f"Latent: {w.n_latent} ({w.n_clued} clued)", cmap="Greys", vmin=0, vmax=1)
    li = H["latent"][i]
    cl = H["clued"][i]
    sx = lambda m: (DY[m] - 1)
    sy = lambda m: (DX[m] - 1)
    ax[4].scatter(sx(li & cl), sy(li & cl), s=10, c="tab:blue", label="clued")
    ax[4].scatter(sx(li & ~cl), sy(li & ~cl), s=10, c="tab:orange", label="unclued")
    ax[4].legend(fontsize=7, loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / f"simulator_example_{DATASET}.png", dpi=150)
plt.show()

In [ ]:
i = examples[0]
z_before = (VALUES[i] - MU) / SD
z_after = (SIM_VALUES[i] - MU) / SD
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
show(axes[0], z_before[:, 0], "Parameter 1 z-score, before injection", cmap="coolwarm", vmin=-8, vmax=8)
show(axes[1], z_after[:, 0], "Parameter 1 z-score, after injection", cmap="coolwarm", vmin=-8, vmax=8)
zmax = np.abs(z_after).max(1)
axes[2].hist(zmax[~SIM_WTF[i]], bins=60, alpha=0.7, label="passing dies")
axes[2].hist(zmax[SIM_WTF[i]], bins=60, alpha=0.7, label="Wafer-Test Fail")
axes[2].axvline(CONFIG["limit_k_sigma"], color="k", ls="--", label="limit (6 sigma)")
axes[2].set_xlabel("max |z| over the 5 parameters")
axes[2].set_yscale("log")
axes[2].legend(fontsize=8)
fig.tight_layout()
plt.show()

## 7. Save outputs
`visible.npz` holds only what a fab would have. `hidden.npz` is the answer key and must never be fed to the model.

In [ ]:
lot_arr = np.repeat(WAFERS.to_numpy(), N_DIE)
wafer_arr = np.repeat(WAFERS.index.to_numpy(), N_DIE)
visible = {"lot": lot_arr.astype(str), "wafer": wafer_arr, "x": np.tile(DX, N_W), "y": np.tile(DY, N_W),
           "wafer_test_fail": SIM_WTF.ravel()}
for j, p in enumerate(PARAMS):
    visible[p] = SIM_VALUES[:, :, j].ravel()
np.savez_compressed(OUT_DIR / "visible.npz", **visible)
np.savez_compressed(OUT_DIR / "hidden.npz", wafer=wafer_arr, x=np.tile(DX, N_W), y=np.tile(DY, N_W),
                    **{kk: vv.ravel() for kk, vv in HIDDEN.items()})
wafers.to_csv(OUT_DIR / "wafers.csv", index=False)
limits.to_json(OUT_DIR / "limits.json", orient="index", indent=2)
(OUT_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))
print("Saved to", OUT_DIR)
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name:20s} {f.stat().st_size / 1e6:6.2f} MB")

In [ ]:
chk = np.load(OUT_DIR / "visible.npz")
allowed = {"lot", "wafer", "x", "y", "wafer_test_fail", *PARAMS}
assert set(chk.files) == allowed, f"visible.npz has unexpected columns: {set(chk.files) - allowed}"
print("visible.npz columns:", chk.files)